# Script of Cobaya that reconstructs P_R(k) from a likelihood and P_monopole(k) data #

In [1]:
# Import required packages
import cobaya
import sys

# Add the path where your local CAMB installation is installed
sys.path.append('/Users/guillermo/Desktop/code/CAMB')

import camb
import numpy as np
import sympy
import math
import matplotlib.pyplot as plt
from scipy.special import erf
from scipy.interpolate import CubicSpline
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import scipy.integrate as integrate

# Cosmological parameters #

In [2]:
# Cosmological constants
c = 2.99792458E5
H = 1/(c/100)

# Cosmological parameters that will not be sampled
OmegakJPAS = 0
gamma = 0.545
kPivot = 0.05  # Mpc^{-1}

# Baseline fixed cosmology from Planck 2018 (TT+TE+EE+lowE+lensing), assuming massless neutrinos
AsJPAS = 2.09052E-9
nsJPAS = 0.9646
tauJPAS = 0.0544
mnuJPAS = 0.0
nmuJPAS = 3.046

# Fiducial values for baseline cosmological parameters that will be sampled
hJPAS = 0.6737
OmegabJPASh2 = 0.02237
OmegaCDMJPASh2 = 0.1200

# Derived cosmological parameters
H0JPAS = hJPAS*100
OmegabJPAS = OmegabJPASh2/hJPAS**2
OmegaCDMJPAS = OmegaCDMJPASh2/hJPAS**2

# LSS parameters #

In [3]:
# Fiducial cosmology functions and constants (including FoG parameter sigmap)
OmegamFid = 0.3136789606771906

# J-PAS Blue sample at z = 0.4 (fourth original redshift bin)
EzFid = 1.2438071021750197
fFid = 0.72647901
XiFid = 1081.8481780362968
DAFid = 772.7486985973549
sigmapFid = 4.7866677867370235

In [4]:
# Linear bias for the J-PAS Blue sample in the z = 0.4 bin
bJPAS = 0.84/0.8097032109405712

In [5]:
# Survey area in square degrees
Asky = 8500

# Fraction of sky covered by the survey
fsky = Asky/(4*np.pi*(180/np.pi)**2)

In [6]:
# Power-law primordial power spectrum. k is in units of h Mpc^{-1}.
def PrimordialPowerLaw(As, ns, k):
    return As*(k/(0.05/hJPAS))**(ns-1)

In [7]:
# Power-law primordial power spectrum with k in units of Mpc^{-1} (without h)
def PrimordialPowerLawSinh(As, ns, k):
    return As*(k/0.05)**(ns-1)

# k and z binning #

In [8]:
# Array limits and steps.
# Extended arrays may be used to estimate derivative quantities.

# k-array limits in units of h Mpc^{-1}
khminKhArrayJPASComplete = 0.001
khmaxKhArrayJPASComplete = 2.4900
stepKhArrayJPASComplete = 0.025

# Full k-binning and reduced k-range (in h Mpc^{-1})
KhArrayJPASComplete = np.exp(
    np.arange(math.log(khminKhArrayJPASComplete),
              math.log(khmaxKhArrayJPASComplete),
              stepKhArrayJPASComplete)
)
KhArrayJPAS = KhArrayJPASComplete[range(120, 212)]

# Upper and lower limits of k-bins
KhArrayJPASUpper = np.zeros(len(KhArrayJPAS))
KhArrayJPASLower = np.zeros(len(KhArrayJPAS))

for i in range(0, len(KhArrayJPAS)-1):
    KhArrayJPASUpper[i] = KhArrayJPAS[i] + (KhArrayJPAS[i+1]-KhArrayJPAS[i])/2
    KhArrayJPASLower[i] = KhArrayJPAS[i] - (KhArrayJPAS[i+1]-KhArrayJPAS[i])/2

# The last element of the upper/lower k arrays can be problematic.
# We copy the penultimate element into the last position.
KhArrayJPASUpper[-1] = KhArrayJPASUpper[-2]
KhArrayJPASLower[-1] = KhArrayJPASLower[-2]

# High-redshift binning in z
zJPASmin = 0.1
zJPASmax = 1.1
stepzJPAS = 0.1

# Original z-bin centers
zJPASPrevious = np.arange(zJPASmin, zJPASmax+stepzJPAS/2, stepzJPAS)

# z-binning including lower and upper edges of all bins
zJPAS = np.arange(
    zJPASmin-stepzJPAS/2,
    zJPASmax+0.01+stepzJPAS/2,
    stepzJPAS/2
)
zJPAS = np.unique(np.append(zJPAS, 0))

# Indices of upper and lower limits of z-bins in the z array
positions_Upper = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23]
positions_Lower = [x - 2 for x in positions_Upper]

# Reading P(k) data and galaxy densities

In [ ]:
# Define a helper function to read the simulated data (P(k), densities),
# and, for checks, the transfer function and the random seed.
# The base path to the data must be provided as input.
def read_data(path_to_data):
    data = {}

    # File name templates
    pk_template = (
        path_to_data +
        'Galaxies_P_g0/JPAS_ForecastDataDeWiggle_Blue_z{:.1f}_8500_CBM_highest_LO.dat'
    )
    densities_file = (
        path_to_data +
        'Densities_And_photoZ_IDR/Blue_Galaxies_CBM_Odds_highest.txt'
    )

    # Initialize arrays
    data['pkz'] = np.zeros((len(zJPASPrevious), len(KhArrayJPAS)))
    data['ndz'] = np.zeros(len(zJPASPrevious))
    data['deltaz'] = np.zeros(len(zJPASPrevious))
    data['vs'] = np.zeros(len(KhArrayJPAS))

    # Loop over redshift bins from z = 0.1 to 1.1 (step 0.1) and read P(k)
    for z_index, z in enumerate(np.arange(0.1, 1.2, 0.1)):
        filename = pk_template.format(z)
        with open(filename) as file:
            for i in range(len(KhArrayJPAS)):
                line = file.readline().split()
                data['pkz'][z_index][i] = float(line[7])
                # Store vs only once (from the first redshift bin)
                if z_index == 0:
                    data['vs'][i] = float(line[6])

    # Read densities and photo-z errors, skipping the header line
    with open(densities_file) as file:
        next(file)  # Skip header
        for i in range(len(zJPASPrevious)):
            line = file.readline().split()
            data['deltaz'][i] = float(line[1])
            data['ndz'][i] = float(line[2])

    return data


# Read data into the dictionary 'data'
# data = read_data('/gpfs/projects/astro/martinezg/J-PAS_Forecast_Data/')
data = read_data('/Users/guillermo/Desktop/J-PAS_Forecast_Data/JPAS_IDR_27deg2/')
data.keys()

# Build an array with the photometric redshift error in the appropriate bins
# for the full extended z-binning.
DeltazBlueJPAS = np.zeros(len(zJPAS))

for i in range(len(zJPASPrevious)):
    DeltazBlueJPAS[2*i + 2] = data['deltaz'][i]

# Cobaya interface classes

In [ ]:
# Define Cobaya theory and external likelihood classes
from cobaya.theory import Theory
from cobaya.likelihood import Likelihood

In [ ]:
# Theory class that modifies the primordial power spectrum P_R(k)
# by introducing spline nodes in log(k)–log(P) space.
class NodesInPrimordialPk(Theory):

    # Initialize with the k-array used for the reconstruction
    def initialize(self):
        self.ks = KhArrayJPAS

    # Compute the modified primordial spectrum from the knot parameters
    def calculate(self, state, want_derived=True, **params_values_dict):

        # Define the k-positions of the PPS nodes (in log-space)
        nodes_logk = [
            (np.log(KhArrayJPAS[-1]) - np.log(KhArrayJPAS[0])) *
            params_values_dict['x1'] + np.log(KhArrayJPAS[0]),
            (np.log(KhArrayJPAS[-1]) - np.log(KhArrayJPAS[0])) *
            params_values_dict['x2'] + np.log(KhArrayJPAS[0])
        ]

        # Define the PPS values at the nodes (in log-space)
        nodes_logPPS = [params_values_dict['y1'], params_values_dict['y2']]

        # Interpolate nodes in log(k)–log(P) space, extrapolating outside the range
        NodesInterpFunc_nodes = interp1d(
            nodes_logk,
            nodes_logPPS,
            kind='linear',
            fill_value='extrapolate'
        )

        # Build the modified PPS(k) evaluated at our k-array.
        # kmin and kmax are given in Mpc^{-1} (without h).
        state['primordial_scalar_pk'] = {
            'kmin': KhArrayJPAS[0]*hJPAS,
            'kmax': KhArrayJPAS[-1]*hJPAS,
            'Pk': np.exp(NodesInterpFunc_nodes(np.log(KhArrayJPAS))),
            'log_regular': True
        }

    # Return the modified primordial scalar power spectrum to Cobaya
    def get_primordial_scalar_pk(self):
        return self.current_state['primordial_scalar_pk']

    # Parameters that this theory can support (to be sampled)
    def get_can_support_params(self):
        return ['x1', 'x2', 'y1', 'y2']

In [ ]:
# Likelihood class that wraps the galaxy power spectrum model and
# computes the monopole and its covariance for the J-PAS Blue sample.
class Pklike(Likelihood):

    def initialize(self):

        # Path to the data products; we call read_data with this base path.
        # self.data = read_data('/gpfs/projects/astro/martinezg/J-PAS_Forecast_Data/')
        self.data = read_data('/Users/guillermo/Desktop/J-PAS_Forecast_Data/JPAS_IDR_27deg2/')

        # Grid of k values
        self.ks = KhArrayJPAS

        # Redshift grid used by the provider
        self.z_win = zJPAS

    def get_requirements(self):
        """
        Declare the quantities that this likelihood needs from Cobaya/CAMB.
        """
        return {
            'omegam': None,
            'Pk_interpolator': {
                'z': self.z_win,
                'k_max': 10,
                'nonlinear': False,
                'vars_pairs': ([['delta_tot', 'delta_tot']])
            },
            'comoving_radial_distance': {'z': self.z_win},
            'angular_diameter_distance': {'z': self.z_win},
            'Hubble': {'z': self.z_win, 'units': 'km/s/Mpc'},
            'sigma8_z': {'z': self.z_win},
            'fsigma8': {'z': self.z_win},
            'CAMBdata': None
        }

    # Definition of the monopole model; it returns:
    #   - the galaxy power spectrum monopole at the target redshift bin
    #   - the covariance of the monopole at the same k-array
    def monopole(self, **params_dic):

        # Printing options for debugging with high precision (if needed)
        np.set_printoptions(precision=24, suppress=True)

        # Access CAMB results through Cobaya
        resultsCobaya = self.provider.get_CAMBdata()

        # Modified primordial power spectrum P(k) evaluated at KhArrayJPAS
        primordialCobaya = self.provider.get_primordial_scalar_pk()

        # Linear matter power spectrum interpolator
        pkCobaya = self.provider.get_Pk_interpolator(
            ('delta_tot', 'delta_tot'),
            nonlinear=False
        )

        # Cosmological parameters from the provider
        Omegam = self.provider.get_param('omegam')

        # Background functions
        Ez = np.sqrt(Omegam*(1+self.z_win)**3 + (1-Omegam))
        HJPAS = H * Ez
        f = (Omegam*(1+self.z_win)**3 / Ez**2)**gamma
        # Comoving radial distance (in Mpc/h) from CAMB
        Xi = self.provider.get_comoving_radial_distance(self.z_win)*hJPAS

        # Angular diameter distance
        DA = Xi/(1+self.z_win)

        # Photometric redshift smearing factor
        sigmar = DeltazBlueJPAS*(1+self.z_win)/HJPAS

        # Fingers-of-God damping at the fiducial redshift bin
        def FFog(mu, k):
            return 1/(1 + (f[8]*k*mu*sigmapFid)**2)

        # Alcock–Paczyński scaling factor
        FactorAP = DAFid**2*Ez[8]/(DA[8]**2*EzFid)

        def Q(mu):
            return (
                (Ez[8]**2*Xi[8]**2*mu**2 -
                 EzFid**2*XiFid**2*(mu**2 - 1))**0.5 /
                (EzFid*Xi[8])
            )

        def muObs(mu):
            return mu*Ez[8]/(EzFid*Q(mu))

        def kObs(mu, k):
            return Q(mu)*k

        # No-wiggle matter power spectrum construction
        KhArrayJPASNoWiggle = KhArrayJPASComplete[120:240]
        FlattenFactor = 1.75

        gradient = np.gradient(
            KhArrayJPASNoWiggle**FlattenFactor *
            pkCobaya.P(self.z_win[8], KhArrayJPASNoWiggle*hJPAS),
            KhArrayJPASNoWiggle
        )

        maxima_indices, _ = find_peaks(gradient)
        maxima_kh = KhArrayJPASNoWiggle[maxima_indices]

        minima_indices, _ = find_peaks(-gradient)
        minima_kh = KhArrayJPASNoWiggle[minima_indices]

        int_order = 2

        Pk_max = pkCobaya.P(self.z_win[8], maxima_kh*hJPAS)
        Pk_min = pkCobaya.P(self.z_win[8], minima_kh*hJPAS)

        max_interp_fun = interp1d(
            maxima_kh,
            maxima_kh**FlattenFactor * Pk_max,
            kind=int_order,
            fill_value="extrapolate"
        )
        min_interp_fun = interp1d(
            minima_kh,
            minima_kh**FlattenFactor * Pk_min,
            kind=int_order,
            fill_value="extrapolate"
        )

        interp_max = max_interp_fun(KhArrayJPASNoWiggle)
        interp_min = min_interp_fun(KhArrayJPASNoWiggle)

        Interp_Maxima = np.array(interp_max)
        Interp_Minima = np.array(interp_min)

        Interp_Mean = (Interp_Maxima + Interp_Minima) / 2
        Interp_Mean_function = interp1d(
            KhArrayJPASNoWiggle,
            Interp_Mean / KhArrayJPASNoWiggle**FlattenFactor,
            kind=int_order,
            fill_value="extrapolate"
        )

        # Linear no-wiggle matter power spectrum
        def PmNoWiggleFunctionLinearJPAS(k):
            return Interp_Mean_function(k)

        exp_factor = 200000

        # Smoothed transition between no‑wiggle and full linear spectrum
        def PmNoWiggleCorrectedFunctionLinearJPAS(k):
            return (
                (1 - np.exp(-exp_factor * k**4)) *
                PmNoWiggleFunctionLinearJPAS(k) +
                np.exp(-exp_factor * k**4) *
                pkCobaya.P(self.z_win[8], k*hJPAS)
            )

        def gmu(mu, k):
            return sigmapFid**2*(1 - mu**2 + mu**2*(1 + fFid)**2)

        # De-wiggled matter power spectrum
        def PmDewiggleJPAS(mu, k):
            return (
                pkCobaya.P(self.z_win[8], k*hJPAS) *
                np.exp(-gmu(mu, k)*k**2) +
                PmNoWiggleCorrectedFunctionLinearJPAS(k) *
                (1 - np.exp(-gmu(mu, k)*k**2))
            )

        # Galaxy power spectrum P_g(mu, k) with AP, FoG and photo-z smearing.
        # The matter spectrum is evaluated at k without h-units.
        def Pg(mu, k):
            return (
                FactorAP *
                FFog(muObs(mu), kObs(mu, k)) *
                (bJPAS + f[8]*muObs(mu)**2)**2 *
                hJPAS**3 *
                PmDewiggleJPAS(muObs(mu), kObs(mu, k)) *
                np.exp(-(kObs(mu, k) * muObs(mu) * sigmar[8])**2)
            )

        # Monopole of the galaxy power spectrum (integration over mu)
        def Pgmonopole(k):
            mu = np.arange(-1, 1, 1/1000)
            return 0.5 * integrate.trapz(Pg(mu, k), mu)

        # Monopole evaluated on our k-array
        PgmonopoleValores = np.zeros(len(KhArrayJPAS))
        for i in range(len(KhArrayJPAS)):
            PgmonopoleValores[i] = Pgmonopole(KhArrayJPAS[i])

        # Covariance of the monopole

        # Comoving distances at lower and upper edges of the z-bins
        XiZaLower = (
            self.provider.get_comoving_radial_distance(
                self.z_win[positions_Lower]
            )*hJPAS
        )
        XiZaUpper = (
            self.provider.get_comoving_radial_distance(
                self.z_win[positions_Upper]
            )*hJPAS
        )

        # Volume between redshift bins
        Vol = 4*np.pi*fsky/3*(XiZaUpper**3 - XiZaLower**3)

        # Number of Fourier modes in a shell between k_inf and k_sup
        def Nk(ksup, kinf):
            return (
                Vol[3] *
                (4*np.pi/3*(ksup**3 - kinf**3)) /
                (2*np.pi)**3
            )

        # Nk evaluated for each k-bin; galaxy densities are read from self.data['ndz']
        NkEvaluado = np.zeros(len(self.ks))
        for i in range(len(self.ks)):
            NkEvaluado[i] = Nk(KhArrayJPASUpper[i], KhArrayJPASLower[i])

        # Integrand for the monopole covariance
        def CovMonopoleIntegral(k):
            mu = np.arange(-1, 1, 1/1000)
            return 0.5 * integrate.trapz(
                (Pg(mu, k) + 1/self.data['ndz'][3])**2,
                mu
            )

        # Covariance integral evaluated on our k-array
        CovMonopoleIntegralValores = np.zeros(len(KhArrayJPAS))
        for i in range(len(KhArrayJPAS)):
            CovMonopoleIntegralValores[i] = CovMonopoleIntegral(KhArrayJPAS[i])

        # Final covariance of the monopole
        CovEvaluado = 2 * CovMonopoleIntegralValores / NkEvaluado

        # Return monopole and covariance at the k-array
        return PgmonopoleValores, CovEvaluado

    # Likelihood evaluation
    def logp(self, **params_values):

        # Allocate arrays for monopole and covariance in all z-bins
        PMonopoleBineado = np.zeros((11, len(self.ks)))
        CovBineado = np.zeros((11, len(self.ks)))

        # Fill the bin corresponding to the Blue sample at z = 0.4
        PMonopoleBineado[3, :len(self.ks)], CovBineado[3, :len(self.ks)] = \
            self.monopole(**params_values)

        # Chi-square–like likelihood with the log of the covariance determinant term
        lnlike = 0.0
        for i in range(len(KhArrayJPAS)):
            lnlike += (
                (PMonopoleBineado[3][i] - data['pkz'][3][i])**2 /
                CovBineado[3][i] +
                np.log(CovBineado[3][i])
            )

        # Return minus half the chi-square
        return -lnlike/2

In [ ]:
# Multivariate Gaussian prior in (ombh2, omch2, H0) based on
# Planck DR3 TT+TE+EE+lowl+lowE+lensing covariance.

# Mean vector for the Gaussian prior
mean_vector = [OmegabJPASh2, OmegaCDMJPASh2, H0JPAS]

# Covariance matrix from Planck DR3 TT+TE+EE+lowl+lowE+lensing
# Order of parameters: Omegab h^2, Omega_c h^2, H0
PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_covMatrix = np.array([
    [2.12418149e-08, -9.03204492e-08, 5.50402459e-05],
    [-9.03204492e-08, 1.38810729e-06, -6.02340614e-04],
    [5.50402459e-05, -6.02340614e-04, 2.86660525e-01]
])

# Determinant of the covariance matrix
determinant = np.linalg.det(
    PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_covMatrix
)

# Inverse covariance matrix (if non-singular)
if determinant != 0:
    inverse_matrix = np.linalg.inv(
        PlanckDR3_base_plikHM_TTTEEE_lowl_lowE_lensing_covMatrix
    )
else:
    print("The covariance matrix is singular and does not have an inverse.")

# Residual vector relative to the fiducial cosmology
def ResidualsVector(ombh2, omch2, H0):
    return np.array([
        ombh2 - OmegabJPASh2,
        omch2 - OmegaCDMJPASh2,
        H0 - H0JPAS
    ])

# Log of the normalized multivariate Gaussian pdf
# (used as an additional prior term).
def multivariate_gaussian_pdf(ombh2, omch2, H0):
    res = ResidualsVector(ombh2, omch2, H0)
    quad_form = np.dot(res.T, np.dot(inverse_matrix, res))
    norm = (2*np.pi)**(-3/2)*determinant**(-1/2)
    return np.log(norm*np.exp(-0.5*quad_form))

In [ ]:
# Cobaya input dictionary. It specifies:
#   - The external likelihood (model + data)
#   - The theory class (primordial PPS with nodes)
#   - Cosmological and node parameters (fixed and sampled)
#   - Sampler settings (PolyChord)
#   - Output directory and additional options.
info = {
    'debug': False,              # Enable debugging information if needed
    'likelihood': {
        'jpass': Pklike          # Attach the custom likelihood class
    },
    'theory': {
        'camb': {"external_primordial_pk": True},
        'my_pk': NodesInPrimordialPk  # Use the nodes-based primordial PPS
    },
    'params': {

        # Fixed cosmological parameters
        'tau': tauJPAS,
        'mnu': mnuJPAS,
        'nnu': nmuJPAS,
        'x1': 0.0,
        'x2': 1.0,
        'ombh2': OmegabJPASh2,
        'omch2': OmegaCDMJPASh2,
        'H0': H0JPAS,

        # Node parameters with flat priors
        'y1': {
            'prior': {'min': -23, 'max': -19},
            'ref': -20,
            'latex': 'y_1'
        },
        'y2': {
            'prior': {'min': -23, 'max': -19},
            'ref': -20,
            'latex': 'y_2'
        },

        # Cosmological parameters to be sampled.
        # The Gaussian priors are defined by (loc, scale).
        'ombh2': {
            'prior': {
                'dist': 'norm',
                'loc': OmegabJPASh2,
                'scale': 0.00015
            },
            'latex': r'\Omega_b h^2'
        },
        'omch2': {
            'prior': {
                'dist': 'norm',
                'loc': OmegaCDMJPASh2,
                'scale': 0.0012
            },
            'latex': r'\Omega_c h^2'
        },
        'H0': {
            'prior': {
                'dist': 'norm',
                'loc': H0JPAS,
                'scale': 0.54
            },
            'latex': r'H_0'
        }
    },

    "sampler": {
        "polychord": {
            "precision_criterion": 1e-1,
            "nlive": 1   # Set to an appropriate value for production runs
        }
    }
}

# Add the multivariate Gaussian prior to the info dict
info["prior"] = {
    "Multivariate": lambda ombh2, omch2, H0: multivariate_gaussian_pdf(
        ombh2, omch2, H0
    )
}

# Path for the output folder and name of the directory.
# Uncomment the path for HPC runs and adjust to your environment.
# info["output"] = (
#     "/gpfs/projects/astro/martinezg/OutputCobaya_JPAS_BlueGalaxies/"
#     "z0p4/LO_8500_CBM_Highest_2Nodes_Results"
# )

info["output"] = (
    "/Users/guillermo/Desktop/JPAS_BlueGalaxies/"
    "z0p4/LO_8500_CBM_Highest/Results"
)

In [ ]:
# Run Cobaya with the specified info dictionary.
# In this notebook, Cobaya is executed directly from Python.
from cobaya import run
updated_info, sampler = run(info)